In [38]:
import pandas as pd

df = pd.read_csv('data/cleaned_members.csv')

y = df['death']
X = df.drop(columns=['death'])

print("death: ", y.shape)
print("data: ", x.shape)

death:  (89000,)
data:  (89000, 29)


loads in data

In [48]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
 X, y, test_size=0.2, stratify=y, random_state = 42)

print(f"Training rows: {X_train.shape[0]}")
print(f"Training rows: {X_test.shape[0]}")


Training rows: 71200
Training rows: 17800
death
False    0.986994
True     0.013006
Name: proportion, dtype: float64


Splits dataset into training and testing set. Stratify is used because there's imbalanced data (1% died)

In [40]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 89000 entries, 0 to 88999
Data columns (total 29 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   peakid         89000 non-null  str    
 1   myear          89000 non-null  int64  
 2   mseason        89000 non-null  str    
 3   sex            89000 non-null  str    
 4   citizen        88993 non-null  str    
 5   status         89000 non-null  str    
 6   residence      80170 non-null  str    
 7   occupation     58261 non-null  str    
 8   leader         89000 non-null  bool   
 9   deputy         89000 non-null  bool   
 10  bconly         89000 non-null  bool   
 11  nottobc        89000 non-null  bool   
 12  support        89000 non-null  bool   
 13  disabled       89000 non-null  bool   
 14  hired          89000 non-null  bool   
 15  sherpa         89000 non-null  bool   
 16  tibetan        89000 non-null  bool   
 17  msolo          89000 non-null  bool   
 18  mtraverse      89

In [41]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = ['age', 'myear']
categorical_features = ['peakid', 'mseason', 'sex', 'citizen', 'status', 'residence', 'occupation', 
                        'leader','deputy', 'bconly', 'nottobc', 'support', 'disabled', 'hired', 'sherpa',
                        'tibetan', 'msolo', 'mtraverse', 'mski', 'mparapente', 'mspeed', 'mo2climb', 'mo2sleep',
                        'mo2medical', 'mo2note', 'route_choice', 'oxygen_status']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

Created numeric and categorical pipeline. Filled missing with median/'missing'. Scaled/One Hot Encoded

In [46]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

dummy = DummyClassifier(strategy="most_frequent")
dummy_pipeline = Pipeline(steps = [('preprocessor', preprocessor), ('classifier', dummy)])

dummy_pipeline.fit(X_train, y_train)
dummy_predictions = dummy_pipeline.predict(X_test)
dummy_probability = dummy_pipeline.predict_proba(X_test)[:,1]

print("--- Dummy Baseline Results ---")
print("Accuracy:", accuracy_score(y_test, dummy_predictions))
print("F1 Score:", f1_score(y_test, dummy_predictions, average='macro'))
print("ROC-AUC:", roc_auc_score(y_test, dummy_probability))

--- Dummy Baseline Results ---
Accuracy: 0.9869662921348314
F1 Score: 0.4967201990499887
ROC-AUC: 0.5


The dummy baseline predicts the most frequent outcome (survived), which gives it an accuracy of 0.98 because most people survived the climb. But it has an F1 score of 49 because its given a lot of false negatives. The F1 score is more realistic as it takes false-positive and negatives into account. The ROC-AUC is .5 which means the model is randomly guessing.

In [51]:
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression(C=1.0, max_iter=1000)

log_reg_pipeline = Pipeline(steps = [('preprocessor', preprocessor),('classifier', log_reg)])

log_reg_pipeline.fit(X_train, y_train)
log_reg_predictions = log_reg_pipeline.predict(X_test)
log_reg_probability = log_reg_pipeline.predict_proba(X_test)[:,1]

print("--- Logistic Regression Baseline Results ---")
print("Accuracy:", accuracy_score(y_test, log_reg_predictions))
print("F1 Score:", f1_score(y_test, log_reg_predictions, average='macro'))
print("ROC-AUC:", roc_auc_score(y_test, log_reg_probability))

--- Logistic Regression Baseline Results ---
Accuracy: 0.9872471910112359
F1 Score: 0.5258357107459363
ROC-AUC: 0.7896761254789273


Logistic Regression actually tries to predict based on the features. The drop in accuracy shows its at least not just guessing the most common outcome, same with the increase in F-Score and ROC-AUC. The F-Score has increased by 3% which isn't a lot but it should grow with the more advanced models. The ROC-AUC has jumped 28% which is really good as there's now a higher chance of the model being correct and it shows its not just guessing based on the most frequent outcome.